# **Pacific Dataviz Challenge 2026**

## **Data collection**

Notebook para baixar somente os datasets selecionados para as frentes de mudancas climaticas.

## **0. Setup**

In [1]:
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import quote
from urllib.request import Request, urlopen
import csv
import json
import shutil
import subprocess

import pandas as pd
from IPython.display import Markdown, display

pd.set_option('display.max_columns', 120)
pd.set_option('display.max_colwidth', 160)


def find_project_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / 'requirements.txt').exists() or (candidate / '.git').exists():
            return candidate
    return cwd


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / 'data'
DOTSTAT_API = 'https://stats-nsi-stable.pacificdata.org/rest/data'
USER_AGENT = 'pacific-dataviz-challenge-26-data-collection/1.0'

FORCE_DOWNLOAD = False
CLEAN_UNSELECTED_DATASETS = True

print(f'PROJECT_ROOT: {PROJECT_ROOT}')
print(f'DATA_DIR: {DATA_DIR}')

PROJECT_ROOT: /Users/gcarninatto/Documents/GitHub/pacific-dataviz-challenge_26
DATA_DIR: /Users/gcarninatto/Documents/GitHub/pacific-dataviz-challenge_26/data


## **1. Selected datasets**

In [2]:
SELECTED_DOTSTAT_DATASETS = [
    {
        'slug': 'mean_surface_temperature_anomalies',
        'title': 'Mean surface temperature anomalies',
        'front': 'land_temperature',
        'agency': 'SPC',
        'dataflow': 'DF_CLIMATE_CHANGE',
        'version': '1.0',
        'key': 'A.ST_ANOM.',
        'source_url': 'https://stats.pacificdata.org/vis?lc=en&df[ds]=SPC2&df[id]=DF_CLIMATE_CHANGE&df[ag]=SPC&df[vs]=1.0&dq=A.ST_ANOM.&pd=,&to[TIME_PERIOD]=false',
    },
    {
        'slug': 'mean_sea_surface_temperature_anomalies',
        'title': 'Mean sea surface temperature anomalies',
        'front': 'sea_temperature',
        'agency': 'SPC',
        'dataflow': 'DF_CLIMATE_CHANGE',
        'version': '1.0',
        'key': 'A.SST_ANOM.',
        'source_url': 'https://stats.pacificdata.org/vis?lc=en&df[ds]=SPC2&df[id]=DF_CLIMATE_CHANGE&df[ag]=SPC&df[vs]=1.0&dq=A.SST_ANOM.&pd=,&to[TIME_PERIOD]=false',
    },
    {
        'slug': 'rainfall_anomalies',
        'title': 'Rainfall anomalies',
        'front': 'rainfall',
        'agency': 'SPC',
        'dataflow': 'DF_CLIMATE_CHANGE',
        'version': '1.0',
        'key': 'A.RAIN_ANOM.',
        'source_url': 'https://stats.pacificdata.org/vis?lc=en&df[ds]=SPC2&df[id]=DF_CLIMATE_CHANGE&df[ag]=SPC&df[vs]=1.0&dq=A.RAIN_ANOM.&pd=,&to[TIME_PERIOD]=false',
    },
    {
        'slug': 'disaster_directly_affected_persons',
        'title': 'Number of directly affected persons attributed to disasters',
        'front': 'disaster_people',
        'agency': 'SPC',
        'dataflow': 'DF_SDG_11',
        'version': '3.0',
        'key': 'A.VC_DSR_AFFCT.........',
        'source_url': 'https://stats.pacificdata.org/vis?lc=en&df[ds]=ds%3ASPC2&df[id]=DF_SDG_11&df[ag]=SPC&df[vs]=3.0&dq=A.VC_DSR_AFFCT.........&pd=,&to[TIME_PERIOD]=false&lb=bt',
    },
    {
        'slug': 'direct_disaster_economic_loss',
        'title': 'Direct disaster economic loss',
        'front': 'disaster_economic_loss',
        'agency': 'SPC',
        'dataflow': 'DF_SDG_11',
        'version': '3.0',
        'key': 'A.VC_DSR_AALT...._T.....',
        'source_url': 'https://stats.pacificdata.org/vis?lc=en&df[ds]=ds%3ASPC2&df[id]=DF_SDG_11&df[ag]=SPC&df[vs]=3.0&dq=A.VC_DSR_AALT...._T.....&pd=,&to[TIME_PERIOD]=false',
    },
    {
        'slug': 'crop_yield',
        'title': 'Crop yield',
        'front': 'food_systems',
        'agency': 'SPC',
        'dataflow': 'DF_CLIMATE_CHANGE',
        'version': '1.0',
        'key': 'A.CROP_YIELD.',
        'source_url': 'https://stats.pacificdata.org/vis?lc=en&df[ds]=SPC2&df[id]=DF_CLIMATE_CHANGE&df[ag]=SPC&df[vs]=1.0&dq=A.CROP_YIELD.&pd=,&to[TIME_PERIOD]=false',
    },
    {
        'slug': 'livestock_yield',
        'title': 'Livestock yield',
        'front': 'food_systems',
        'agency': 'SPC',
        'dataflow': 'DF_CLIMATE_CHANGE',
        'version': '1.0',
        'key': 'A.LVST_YIELD.',
        'source_url': 'https://stats.pacificdata.org/vis?lc=en&df[ds]=SPC2&df[id]=DF_CLIMATE_CHANGE&df[ag]=SPC&df[vs]=1.0&dq=A.LVST_YIELD.&pd=,&to[TIME_PERIOD]=false',
    },
    {
        'slug': 'agricultural_yield_disaggregated',
        'title': 'Agricultural yield - disaggregated',
        'front': 'food_systems',
        'agency': 'SPC',
        'dataflow': 'DF_AGRICULTURAL_PRODUCTION',
        'version': '1.0',
        'key': 'A...',
        'source_url': 'https://stats.pacificdata.org/vis?lc=en&df[ds]=SPC2&df[id]=DF_AGRICULTURAL_PRODUCTION&df[ag]=SPC&df[vs]=1.0&dq=A...&pd=,&to[TIME_PERIOD]=false',
    },
]


def dotstat_url(dataset: dict[str, str]) -> str:
    flow = ','.join([dataset['agency'], dataset['dataflow'], dataset['version']])
    key = quote(dataset['key'], safe='.+_')
    return f'{DOTSTAT_API}/{flow}/{key}?dimensionAtObservation=AllDimensions&format=csv'


catalog = pd.DataFrame(SELECTED_DOTSTAT_DATASETS)
catalog['api_url'] = catalog.apply(lambda row: dotstat_url(row.to_dict()), axis=1)
display(catalog[['slug', 'front', 'title', 'dataflow', 'key']])

,slug,front,title,dataflow,key
0,mean_surface_temperature_anomalies,land_temperature,Mean surface temperature anomalies,DF_CLIMATE_CHANGE,A.ST_ANOM.
1,mean_sea_surface_temperature_anomalies,sea_temperature,Mean sea surface temperature anomalies,DF_CLIMATE_CHANGE,A.SST_ANOM.
2,rainfall_anomalies,rainfall,Rainfall anomalies,DF_CLIMATE_CHANGE,A.RAIN_ANOM.
3,disaster_directly_affected_persons,disaster_people,Number of directly affected persons attributed to disasters,DF_SDG_11,A.VC_DSR_AFFCT.........
4,direct_disaster_economic_loss,disaster_economic_loss,Direct disaster economic loss,DF_SDG_11,A.VC_DSR_AALT...._T.....
5,crop_yield,food_systems,Crop yield,DF_CLIMATE_CHANGE,A.CROP_YIELD.
6,livestock_yield,food_systems,Livestock yield,DF_CLIMATE_CHANGE,A.LVST_YIELD.
7,agricultural_yield_disaggregated,food_systems,Agricultural yield - disaggregated,DF_AGRICULTURAL_PRODUCTION,A...


## **2. Download helpers**

In [3]:
def run_curl(url: str, target: Path) -> None:
    target.parent.mkdir(parents=True, exist_ok=True)
    cmd = [
        'curl', '-L', '--fail', '--silent', '--show-error',
        '--retry', '4', '--retry-delay', '2',
        '--user-agent', USER_AGENT,
        '--output', str(target), url,
    ]
    subprocess.run(cmd, check=True)


def download_text(url: str, target: Path) -> None:
    target.parent.mkdir(parents=True, exist_ok=True)
    request = Request(url, headers={'User-Agent': USER_AGENT, 'Accept-Language': 'en'})
    with urlopen(request, timeout=120) as response:
        target.write_bytes(response.read())


def download_file(url: str, target: Path, *, force: bool = False) -> None:
    if target.exists() and target.stat().st_size > 0 and not force:
        print(f'skip existing {target.relative_to(PROJECT_ROOT)}')
        return
    print(f'download {target.relative_to(PROJECT_ROOT)}')
    if shutil.which('curl'):
        run_curl(url, target)
    else:
        download_text(url, target)


def write_json(path: Path, payload: dict) -> None:
    path.write_text(json.dumps(payload, indent=2, sort_keys=True) + '\n', encoding='utf-8')


def count_csv_rows(path: Path) -> int:
    with path.open('r', encoding='utf-8', newline='') as handle:
        return max(sum(1 for _ in handle) - 1, 0)


def file_size(path: Path) -> int:
    return path.stat().st_size if path.exists() else 0


def clean_unselected_data() -> None:
    keep = {dataset['slug'] for dataset in SELECTED_DOTSTAT_DATASETS}
    keep.update({'README.md', 'manifest.csv'})
    if not DATA_DIR.exists():
        return
    for path in DATA_DIR.iterdir():
        if path.name in keep:
            continue
        if path.is_dir():
            print(f'remove stale dataset {path.relative_to(PROJECT_ROOT)}')
            shutil.rmtree(path)
        elif path.is_file():
            print(f'remove stale file {path.relative_to(PROJECT_ROOT)}')
            path.unlink()


def write_readme() -> None:
    lines = [
        '# Pacific Dataviz Challenge 2026 Data',
        '',
        'Selected official datasets for the current climate-change exploration.',
        '',
        'Current fronts:',
        '',
        '- land temperature anomalies',
        '- sea surface temperature anomalies',
        '- rainfall anomalies',
        '- people directly affected by disasters',
        '- direct economic loss from disasters',
        '- crop and livestock yield as food-system context',
        '',
        'Each dataset folder contains `data.csv` and `metadata.json`.',
        '',
        'Sources:',
        '',
        '- Competition page: https://pacificdatavizchallenge.org/',
        '- .Stat API endpoint: https://stats-nsi-stable.pacificdata.org',
        '',
    ]
    (DATA_DIR / 'README.md').write_text('\n'.join(lines), encoding='utf-8')

## **3. Run data collection**

In [4]:
DATA_DIR.mkdir(parents=True, exist_ok=True)

if CLEAN_UNSELECTED_DATASETS:
    clean_unselected_data()

downloaded_at = datetime.now(timezone.utc).replace(microsecond=0).isoformat()
manifest_rows = []

for dataset in SELECTED_DOTSTAT_DATASETS:
    dataset_dir = DATA_DIR / dataset['slug']
    dataset_dir.mkdir(parents=True, exist_ok=True)

    api_url = dotstat_url(dataset)
    data_file = dataset_dir / 'data.csv'
    download_file(api_url, data_file, force=FORCE_DOWNLOAD)

    metadata = {
        'api_url': api_url,
        'dataflow': dataset['dataflow'],
        'downloaded_at_utc': downloaded_at,
        'files': [{'path': 'data.csv', 'size_bytes': file_size(data_file)}],
        'front': dataset['front'],
        'key': dataset['key'],
        'source_url': dataset['source_url'],
        'title': dataset['title'],
        'type': 'dotstat_csv',
    }
    write_json(dataset_dir / 'metadata.json', metadata)

    manifest_rows.append({
        'slug': dataset['slug'],
        'title': dataset['title'],
        'front': dataset['front'],
        'type': 'dotstat_csv',
        'local_files': 'data.csv; metadata.json',
        'size_bytes': file_size(data_file),
        'rows': count_csv_rows(data_file),
        'source_url': dataset['source_url'],
        'api_url': api_url,
    })

manifest = pd.DataFrame(manifest_rows)
manifest.to_csv(DATA_DIR / 'manifest.csv', index=False)
write_readme()

display(manifest[['slug', 'front', 'title', 'rows', 'size_bytes']])

download data/mean_surface_temperature_anomalies/data.csv
download data/mean_sea_surface_temperature_anomalies/data.csv
download data/rainfall_anomalies/data.csv
download data/disaster_directly_affected_persons/data.csv
download data/direct_disaster_economic_loss/data.csv
download data/crop_yield/data.csv
download data/livestock_yield/data.csv
download data/agricultural_yield_disaggregated/data.csv


,slug,front,title,rows,size_bytes
0,mean_surface_temperature_anomalies,land_temperature,Mean surface temperature anomalies,3872,266883
1,mean_sea_surface_temperature_anomalies,sea_temperature,Mean sea surface temperature anomalies,3696,258217
2,rainfall_anomalies,rainfall,Rainfall anomalies,1034,68908
3,disaster_directly_affected_persons,disaster_people,Number of directly affected persons attributed to disasters,174,69824
4,direct_disaster_economic_loss,disaster_economic_loss,Direct disaster economic loss,39,12653
5,crop_yield,food_systems,Crop yield,900,59413
6,livestock_yield,food_systems,Livestock yield,866,57689
7,agricultural_yield_disaggregated,food_systems,Agricultural yield - disaggregated,20725,1649599


## **4. Read each dataset individually**

In [5]:
MEAN_SURFACE_TEMPERATURE_ANOMALIES = pd.read_csv(DATA_DIR / 'mean_surface_temperature_anomalies' / 'data.csv')
display(Markdown('### **Mean surface temperature anomalies**'))
display(MEAN_SURFACE_TEMPERATURE_ANOMALIES.head())

### **Mean surface temperature anomalies**

,DATAFLOW,FREQ,CLIMATE_CHANGE_INDICATORS,GEO_PICT,TIME_PERIOD,OBS_VALUE,OBS_STATUS,UNIT_MEASURE,ERROR_TYPE,ERROR_VAL,REPORTING_TYPE,OBS_COMMENT
0,SPC:DF_CLIMATE_CHANGE(1.0),A,ST_ANOM,WF,1850,0.0,NaN,CELSIUS,SE,0.0,N,NaN
1,SPC:DF_CLIMATE_CHANGE(1.0),A,ST_ANOM,WF,1851,-0.4,NaN,CELSIUS,SE,0.0,N,NaN
2,SPC:DF_CLIMATE_CHANGE(1.0),A,ST_ANOM,WF,1852,-0.1,NaN,CELSIUS,SE,0.0,N,NaN
3,SPC:DF_CLIMATE_CHANGE(1.0),A,ST_ANOM,WF,1853,0.1,NaN,CELSIUS,SE,0.1,N,NaN
4,SPC:DF_CLIMATE_CHANGE(1.0),A,ST_ANOM,WF,1854,-0.1,NaN,CELSIUS,SE,0.1,N,NaN


In [6]:
MEAN_SEA_SURFACE_TEMPERATURE_ANOMALIES = pd.read_csv(DATA_DIR / 'mean_sea_surface_temperature_anomalies' / 'data.csv')
display(Markdown('### **Mean sea surface temperature anomalies**'))
display(MEAN_SEA_SURFACE_TEMPERATURE_ANOMALIES.head())

### **Mean sea surface temperature anomalies**

,DATAFLOW,FREQ,CLIMATE_CHANGE_INDICATORS,GEO_PICT,TIME_PERIOD,OBS_VALUE,OBS_STATUS,UNIT_MEASURE,ERROR_TYPE,ERROR_VAL,REPORTING_TYPE,OBS_COMMENT
0,SPC:DF_CLIMATE_CHANGE(1.0),A,SST_ANOM,AS,1850,0.0,NaN,CELSIUS,SE,0.0,N,NaN
1,SPC:DF_CLIMATE_CHANGE(1.0),A,SST_ANOM,AS,1851,-0.3,NaN,CELSIUS,SE,0.0,N,NaN
2,SPC:DF_CLIMATE_CHANGE(1.0),A,SST_ANOM,AS,1852,0.0,NaN,CELSIUS,SE,0.0,N,NaN
3,SPC:DF_CLIMATE_CHANGE(1.0),A,SST_ANOM,AS,1853,0.4,NaN,CELSIUS,SE,0.1,N,NaN
4,SPC:DF_CLIMATE_CHANGE(1.0),A,SST_ANOM,AS,1854,0.2,NaN,CELSIUS,SE,0.1,N,NaN


In [7]:
RAINFALL_ANOMALIES = pd.read_csv(DATA_DIR / 'rainfall_anomalies' / 'data.csv')
display(Markdown('### **Rainfall anomalies**'))
display(RAINFALL_ANOMALIES.head())

### **Rainfall anomalies**

,DATAFLOW,FREQ,CLIMATE_CHANGE_INDICATORS,GEO_PICT,TIME_PERIOD,OBS_VALUE,OBS_STATUS,UNIT_MEASURE,ERROR_TYPE,ERROR_VAL,REPORTING_TYPE,OBS_COMMENT
0,SPC:DF_CLIMATE_CHANGE(1.0),A,RAIN_ANOM,TK,1979,-14.2,NaN,MM,SE,0.7,N,NaN
1,SPC:DF_CLIMATE_CHANGE(1.0),A,RAIN_ANOM,TK,1980,-8.3,NaN,MM,SE,0.8,N,NaN
2,SPC:DF_CLIMATE_CHANGE(1.0),A,RAIN_ANOM,TK,1981,6.1,NaN,MM,SE,1.3,N,NaN
3,SPC:DF_CLIMATE_CHANGE(1.0),A,RAIN_ANOM,TK,1982,-5.5,NaN,MM,SE,1.1,N,NaN
4,SPC:DF_CLIMATE_CHANGE(1.0),A,RAIN_ANOM,TK,1983,14.7,NaN,MM,SE,1.5,N,NaN


In [8]:
DISASTER_DIRECTLY_AFFECTED_PERSONS = pd.read_csv(DATA_DIR / 'disaster_directly_affected_persons' / 'data.csv')
display(Markdown('### **Disaster directly affected persons**'))
display(DISASTER_DIRECTLY_AFFECTED_PERSONS.head())

### **Disaster directly affected persons**

,DATAFLOW,FREQ,INDICATOR,GEO_PICT,SEX,AGE,URBANIZATION,INCOME,EDUCATION,OCCUPATION,COMPOSITE_BREAKDOWN,DISABILITY,TIME_PERIOD,OBS_VALUE,UNIT_MEASURE,REPORTING_TYPE,NATURE,DATA_SOURCE,OBS_STATUS,OBS_COMMENT
0,SPC:DF_SDG_11(3.0),A,VC_DSR_AFFCT,NC,_T,_T,_T,_T,_T,_T,_Z,_T,2009,0,N,G,C,United Nations Office for Disaster Risk Reduction (2022),NaN,Obs_Value = 0; Disclaimer: The data being submitted by UNDRR has been extracted from two sources: (a) the Sendai Framework Monitoring System as provided by...
1,SPC:DF_SDG_11(3.0),A,VC_DSR_AFFCT,NC,_T,_T,_T,_T,_T,_T,_Z,_T,2011,0,N,G,C,United Nations Office for Disaster Risk Reduction (2022),NaN,Obs_Value = 0; Disclaimer: The data being submitted by UNDRR has been extracted from two sources: (a) the Sendai Framework Monitoring System as provided by...
2,SPC:DF_SDG_11(3.0),A,VC_DSR_AFFCT,NC,_T,_T,_T,_T,_T,_T,_Z,_T,2017,4,N,G,C,United Nations Office for Disaster Risk Reduction (2022),NaN,Disclaimer: The data being submitted by UNDRR has been extracted from two sources: (a) the Sendai Framework Monitoring System as provided by designated nati...
3,SPC:DF_SDG_11(3.0),A,VC_DSR_AFFCT,NC,_T,_T,_T,_T,_T,_T,_Z,_T,2018,0,N,G,C,United Nations Office for Disaster Risk Reduction (2022),NaN,Obs_Value = 0; Disclaimer: The data being submitted by UNDRR has been extracted from two sources: (a) the Sendai Framework Monitoring System as provided by...
4,SPC:DF_SDG_11(3.0),A,VC_DSR_AFFCT,NC,_T,_T,_T,_T,_T,_T,_Z,_T,2019,1,N,G,C,United Nations Office for Disaster Risk Reduction (2022),NaN,Disclaimer: The data being submitted by UNDRR has been extracted from two sources: (a) the Sendai Framework Monitoring System as provided by designated nati...


In [9]:
DIRECT_DISASTER_ECONOMIC_LOSS = pd.read_csv(DATA_DIR / 'direct_disaster_economic_loss' / 'data.csv')
display(Markdown('### **Direct disaster economic loss**'))
display(DIRECT_DISASTER_ECONOMIC_LOSS.head())

### **Direct disaster economic loss**

,DATAFLOW,FREQ,INDICATOR,GEO_PICT,SEX,AGE,URBANIZATION,INCOME,EDUCATION,OCCUPATION,COMPOSITE_BREAKDOWN,DISABILITY,TIME_PERIOD,OBS_VALUE,UNIT_MEASURE,REPORTING_TYPE,NATURE,DATA_SOURCE,OBS_STATUS,OBS_COMMENT
0,SPC:DF_SDG_11(3.0),A,VC_DSR_AALT,KI,_T,_T,_T,_T,_T,_T,_Z,_T,2014,268950.0,USD,G,C,United Nations Office for Disaster Risk Reduction (2021),NaN,Disclaimer: the data submitted by UNISDR has been extracted from 2 sources: (a) the Sendai Framework Monitoring System as provided by designated national fo...
1,SPC:DF_SDG_11(3.0),A,VC_DSR_AALT,KI,_T,_T,_T,_T,_T,_T,_Z,_T,2020,30750.0,USD,G,C,United Nations Office for Disaster Risk Reduction (2021),NaN,Disclaimer: the data submitted by UNISDR has been extracted from 2 sources: (a) the Sendai Framework Monitoring System as provided by designated national fo...
2,SPC:DF_SDG_11(3.0),A,VC_DSR_AALT,WS,_T,_T,_T,_T,_T,_T,_Z,_T,2016,22048411.0,USD,G,C,UNISDR (2018). Sendai Framework Monitor - United Nations Office for Disaster Risk Reduction,NaN,NaN
3,SPC:DF_SDG_11(3.0),A,VC_DSR_AALT,FM,_T,_T,_T,_T,_T,_T,_Z,_T,2016,20310000.0,USD,G,C,UNISDR (2018). Sendai Framework Monitor - United Nations Office for Disaster Risk Reduction,NaN,NaN
4,SPC:DF_SDG_11(3.0),A,VC_DSR_AALT,FM,_T,_T,_T,_T,_T,_T,_Z,_T,2018,196875.0,USD,G,C,United Nations Office for Disaster Risk Reduction (2021),NaN,Disclaimer: the data submitted by UNISDR has been extracted from 2 sources: (a) the Sendai Framework Monitoring System as provided by designated national fo...


In [10]:
CROP_YIELD = pd.read_csv(DATA_DIR / 'crop_yield' / 'data.csv')
display(Markdown('### **Crop yield**'))
display(CROP_YIELD.head())

### **Crop yield**

,DATAFLOW,FREQ,CLIMATE_CHANGE_INDICATORS,GEO_PICT,TIME_PERIOD,OBS_VALUE,OBS_STATUS,UNIT_MEASURE,ERROR_TYPE,ERROR_VAL,REPORTING_TYPE,OBS_COMMENT
0,SPC:DF_CLIMATE_CHANGE(1.0),A,CROP_YIELD,VU,1961,3723.4,NaN,KGHA,NaN,NaN,NaN,NaN
1,SPC:DF_CLIMATE_CHANGE(1.0),A,CROP_YIELD,VU,1962,3661.5,NaN,KGHA,NaN,NaN,NaN,NaN
2,SPC:DF_CLIMATE_CHANGE(1.0),A,CROP_YIELD,VU,1963,4055.0,NaN,KGHA,NaN,NaN,NaN,NaN
3,SPC:DF_CLIMATE_CHANGE(1.0),A,CROP_YIELD,VU,1964,4312.0,NaN,KGHA,NaN,NaN,NaN,NaN
4,SPC:DF_CLIMATE_CHANGE(1.0),A,CROP_YIELD,VU,1965,3499.0,NaN,KGHA,NaN,NaN,NaN,NaN


In [11]:
LIVESTOCK_YIELD = pd.read_csv(DATA_DIR / 'livestock_yield' / 'data.csv')
display(Markdown('### **Livestock yield**'))
display(LIVESTOCK_YIELD.head())

### **Livestock yield**

,DATAFLOW,FREQ,CLIMATE_CHANGE_INDICATORS,GEO_PICT,TIME_PERIOD,OBS_VALUE,OBS_STATUS,UNIT_MEASURE,ERROR_TYPE,ERROR_VAL,REPORTING_TYPE,OBS_COMMENT
0,SPC:DF_CLIMATE_CHANGE(1.0),A,LVST_YIELD,TV,1961,2104.6,NaN,KG_AN,NaN,NaN,NaN,NaN
1,SPC:DF_CLIMATE_CHANGE(1.0),A,LVST_YIELD,TV,1962,2109.3,NaN,KG_AN,NaN,NaN,NaN,NaN
2,SPC:DF_CLIMATE_CHANGE(1.0),A,LVST_YIELD,TV,1963,2113.9,NaN,KG_AN,NaN,NaN,NaN,NaN
3,SPC:DF_CLIMATE_CHANGE(1.0),A,LVST_YIELD,TV,1964,2257.4,NaN,KG_AN,NaN,NaN,NaN,NaN
4,SPC:DF_CLIMATE_CHANGE(1.0),A,LVST_YIELD,TV,1965,2259.7,NaN,KG_AN,NaN,NaN,NaN,NaN


In [12]:
AGRICULTURAL_YIELD_DISAGGREGATED = pd.read_csv(DATA_DIR / 'agricultural_yield_disaggregated' / 'data.csv')
display(Markdown('### **Agricultural yield - disaggregated**'))
display(AGRICULTURAL_YIELD_DISAGGREGATED.head())

### **Agricultural yield - disaggregated**

,DATAFLOW,FREQ,GEO_PICT,AGRICULTURE_PRODUCTION_ITEM,AGRICULTURE_PRODUCTION_TYPE,TIME_PERIOD,OBS_VALUE,OBS_STATUS,UNIT_MEASURE,ERROR_TYPE,ERROR_VAL,REPORTING_TYPE,OBS_COMMENT
0,SPC:DF_AGRICULTURAL_PRODUCTION(1.0),A,SB,COCOA,CROP_YIELD,1961,500.0,NaN,KGHA,NaN,NaN,NaN,NaN
1,SPC:DF_AGRICULTURAL_PRODUCTION(1.0),A,SB,COCOA,CROP_YIELD,1962,480.0,NaN,KGHA,NaN,NaN,NaN,NaN
2,SPC:DF_AGRICULTURAL_PRODUCTION(1.0),A,SB,COCOA,CROP_YIELD,1963,500.0,NaN,KGHA,NaN,NaN,NaN,NaN
3,SPC:DF_AGRICULTURAL_PRODUCTION(1.0),A,SB,COCOA,CROP_YIELD,1964,512.0,NaN,KGHA,NaN,NaN,NaN,NaN
4,SPC:DF_AGRICULTURAL_PRODUCTION(1.0),A,SB,COCOA,CROP_YIELD,1965,500.0,NaN,KGHA,NaN,NaN,NaN,NaN
